# 01 — Python + EO orientation
## Galičica National Park case study

**Goal:** get comfortable with the course workflow: **Jupyter + Python as the interface, Earth Engine as a cloud EO backend**.

By the end you will have:
- defined the Galičica study area;
- loaded and filtered Sentinel-2 imagery;
- built a cloud-masked composite;
- displayed true-colour and SWIR false-colour imagery;
- calculated **NDVI** and **NBR**;
- extracted simple statistics.

This is **not a Python programming lesson**. Most code is already written. Your job is to run it, change a few parameters, and interpret the result.

### How to work with this notebook

- Run cells from top to bottom with **Shift + Enter**.
- Change only values marked **TRY THIS** unless you know what you are doing.
- If a live backend fails, tell a trainer rather than spending half the session debugging.

## 1. Imports and Earth Engine

In [ ]:
from pathlib import Path
import ee
import pandas as pd
import geopandas as gpd
import folium

GEE_PROJECT_ID = "ee-andreydara"

try:
    ee.Initialize(project=GEE_PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)

print("Earth Engine ready:", ee.Number(2).add(3).getInfo() == 5)

## 2. Define the study area

We use one **canonical Galičica course AOI** throughout the practicals. It is stored in:

`data/aoi/galicica_aoi.geojson`

The cell below finds it automatically whether the notebook is launched from the repository root or the `notebooks/` folder. If the file is missing, it falls back to the earlier rectangular training AOI.

In [ ]:
repo_root = Path.home() / "mystorage" / "fire-school"

aoi_candidates = [
    repo_root / "data" / "aoi" / "galicica_aoi.geojson",
    Path.cwd() / "data" / "aoi" / "galicica_aoi.geojson",
    Path.cwd().parent / "data" / "aoi" / "galicica_aoi.geojson",
]

AOI_PATH = next((p for p in aoi_candidates if p.exists()), None)
AOI_GDF = None

if AOI_PATH is not None:
    AOI_GDF = gpd.read_file(AOI_PATH).to_crs("EPSG:4326")
    aoi_geom = AOI_GDF.geometry.iloc[0]
    AOI = ee.Geometry(aoi_geom.__geo_interface__)
    centroid = aoi_geom.centroid
    CENTER = [centroid.y, centroid.x]
    print("Using canonical AOI:", AOI_PATH)
else:
    AOI = ee.Geometry.Rectangle([20.78, 40.86, 21.12, 41.18])
    CENTER = [41.02, 20.95]
    print("Canonical AOI file not found — using rectangular fallback.")

ZOOM = 10

print("AOI area (km²):", round(AOI.area().divide(1e6).getInfo(), 1))

## 3. Choose a period and load Sentinel-2

In [ ]:
# TRY THIS: change the dates later and rerun the cells below.
START_DATE = "2024-06-01"
END_DATE   = "2024-07-31"
MAX_CLOUD  = 40

s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(AOI)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_CLOUD))
)

print("Sentinel-2 scenes:", s2.size().getInfo())

## 4. Mask clouds and make a median composite

Sentinel-2 Level-2A contains a **Scene Classification Layer (SCL)**.  
Here we mask cloud shadow, clouds, cirrus and snow/ice, then calculate a median composite.

The exact masking strategy can matter in real analyses; this one is deliberately simple for teaching.

In [ ]:
def mask_s2_scl(img):
    scl = img.select("SCL")
    bad = (
        scl.eq(3)   # cloud shadow
        .Or(scl.eq(8))   # cloud medium probability
        .Or(scl.eq(9))   # cloud high probability
        .Or(scl.eq(10))  # cirrus
        .Or(scl.eq(11))  # snow / ice
    )

    # Keep only bands used in this practical.
    # This also guarantees a homogeneous collection if auxiliary
    # Sentinel-2 bands differ between processing baselines.
    return (
        img.updateMask(bad.Not())
        .select(["B2", "B3", "B4", "B8", "B11", "B12"])
    )

composite = s2.map(mask_s2_scl).median().clip(AOI)

print("Composite bands:", composite.bandNames().getInfo())

## 5. Display the imagery

In [ ]:
def add_ee_layer(m, ee_image, vis_params, name):
    map_id = ee_image.getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id["tile_fetcher"].url_format,
        attr="Google Earth Engine",
        name=name,
        overlay=True,
        control=True,
    ).add_to(m)

m = folium.Map(location=CENTER, zoom_start=ZOOM, tiles="CartoDB positron")

if AOI_GDF is not None:
    folium.GeoJson(
        AOI_GDF,
        name="Course AOI",
        style_function=lambda _: {
            "color": "black",
            "weight": 2,
            "fillOpacity": 0.0,
        },
    ).add_to(m)

add_ee_layer(
    m, composite,
    {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000},
    "Sentinel-2 true colour"
)

add_ee_layer(
    m, composite,
    {"bands": ["B12", "B8", "B4"], "min": 0, "max": 3500},
    "SWIR / NIR / Red"
)

folium.LayerControl().add_to(m)
m

### Think before moving on

1. Which surface types stand out in true colour?
2. What changes in the SWIR/NIR/Red view?
3. Why might SWIR be useful for wildfire applications?

## 6. Calculate NDVI and NBR

In [ ]:
ndvi = composite.normalizedDifference(["B8", "B4"]).rename("NDVI")
nbr  = composite.normalizedDifference(["B8", "B12"]).rename("NBR")

print("NDVI and NBR calculated.")

In [ ]:
m2 = folium.Map(location=CENTER, zoom_start=ZOOM, tiles="CartoDB positron")

add_ee_layer(
    m2, ndvi,
    {"min": -0.2, "max": 0.9, "palette": ["8c510a", "f6e8c3", "01665e"]},
    "NDVI"
)

add_ee_layer(
    m2, nbr,
    {"min": -0.5, "max": 0.9, "palette": ["b2182b", "f7f7f7", "2166ac"]},
    "NBR"
)

folium.LayerControl().add_to(m2)
m2

**Interpretation**

- **NDVI** contrasts NIR and red reflectance and is commonly used as an indicator of green vegetation.
- **NBR** contrasts NIR and SWIR2. Fire strongly changes both vegetation structure and SWIR response, which makes NBR useful for burn assessment.

Neither index directly measures biodiversity, fuel mass, or fire probability.

## 7. Simple regional statistics

In [ ]:
stats = ee.Image.cat([ndvi, nbr]).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=AOI,
    scale=100,
    maxPixels=1e8,
    bestEffort=True,
).getInfo()

pd.DataFrame([stats]).round(3)

## 8. Mini challenge — 10 minutes

Choose **one**:

1. Change the date range to **May–June 2024** and compare the composite.
2. Tighten `MAX_CLOUD` from `40` to `10`. How many scenes remain?
3. Add **NDMI** using Sentinel-2 bands `B8` and `B11`.
4. Click around the two maps and identify where NBR behaves differently from NDVI.

### Take-away

You have already used the workflow we will repeat all week:

**question → AOI → image collection → filtering/masking → derived variable → map/statistics → interpretation**

## Optional: save your notebook

Keep course work under `~/mystorage` in CDSE JupyterLab because that directory is persistent.